In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Thiết lập cấu trúc đường dẫn dự án
drive_path = '/content/drive/MyDrive/FMNN_CODE/FMN_EduWarning'
raw_data_path = os.path.join(drive_path, 'datasets/raw')
processed_data_path = os.path.join(drive_path, 'datasets/processed')
result_path = os.path.join(drive_path, 'results')
figure_path = os.path.join(drive_path, 'figures')

# Tạo các thư mục nếu chưa tồn tại
for p in [raw_data_path, processed_data_path, result_path, figure_path]:
    os.makedirs(p, exist_ok=True)

print(f"✅ Đã kết nối Drive và trỏ dữ liệu tại: {raw_data_path}")

Mounted at /content/drive
✅ Đã kết nối Drive và trỏ dữ liệu tại: /content/drive/MyDrive/FMNN_CODE/FMN_EduWarning/datasets/raw


---

#### Bước 2: Script chính thực thi mô hình (`fmnn_module_test_v2.py`)

Đoạn code dưới đây tự động dò tìm file `ket_qua_xet_len_lop_ktpm.csv` theo thứ tự ưu tiên: từ thư mục `raw_data_path` trên Google Drive, thư mục làm việc hiện tại, hoặc không gian lưu trữ trực tuyến:

In [2]:
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional, Union
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# =============================================================================
# 1. LỚP HYPERBOX - HỘP SIÊU KHÔNG GIAN MỜ (Đại diện cho các vùng luật quyết định)
# =============================================================================
class Hyperbox:
    def __init__(self, box_id: int, x: np.ndarray, class_label: Union[int, str]):
        self.box_id = box_id
        # Tọa độ cực tiểu (v) và cực đại (w) ban đầu trùng với giá trị của mẫu x đầu tiên
        self.v = np.asarray(x, dtype=float).copy()
        self.w = np.asarray(x, dtype=float).copy()
        self.class_label = class_label
        self.expanded_count = 1  # Đếm số lượng sinh viên thuộc phạm vi bao phủ của hộp

    @property
    def n_features(self) -> int:
        # Trả về số lượng đặc trưng đầu vào (ở đây là 5 đặc trưng học tập)
        return len(self.v)

    @property
    def center(self) -> np.ndarray:
        # Tìm điểm trung tâm của hộp siêu không gian mờ
        return (self.v + self.w) / 2.0

    def calculate_membership(self, x: np.ndarray, gamma: float = 1.0) -> float:
        """
        Tính độ khớp (độ thuộc mờ) của sinh viên x đối với hộp mờ hiện tại.
        Giá trị trả về nằm trong khoảng [0, 1]. Số càng lớn càng thuộc về hộp.
        """
        x = np.asarray(x, dtype=float)

        # Tính toán mức độ vượt biên của điểm x so với cận trên (w) và cận dưới (v)
        upper_diff = gamma * (x - self.w)
        lower_diff = gamma * (self.v - x)

        # Áp dụng hàm dốc (ramp) để giới hạn giá trị trong khoảng [0, 1]
        upper_violation = np.minimum(1.0, np.maximum(0.0, upper_diff))
        lower_violation = np.minimum(1.0, np.maximum(0.0, lower_diff))

        # Tổng hợp mức độ thuộc mờ trung bình trên tất cả các chiều đặc trưng
        total_dim_membership = 0.0
        for i in range(self.n_features):
            dim_score = (1.0 - upper_violation[i]) + (1.0 - lower_violation[i])
            total_dim_membership += dim_score

        membership_score = total_dim_membership / (2.0 * self.n_features)
        return float(membership_score)

    def can_expand(self, x: np.ndarray, theta: float, epsilon: float = 1e-12) -> bool:
        """
        Kiểm tra xem hộp có thể phình to để nuốt thêm điểm dữ liệu x hay không.
        Đảm bảo tổng kích thước các chiều của hộp sau khi phình không vượt quá giới hạn (theta).
        """
        x = np.asarray(x, dtype=float)

        # Dự đoán kích thước hộp mới nếu gom thêm điểm x
        temp_v = np.minimum(self.v, x)
        temp_w = np.maximum(self.w, x)
        new_size_sum = float(np.sum(temp_w - temp_v))

        # Kiểm tra điều kiện giới hạn kích thước siêu hộp mờ
        limit = self.n_features * theta
        return new_size_sum <= (limit + epsilon)

    def expand(self, x: np.ndarray) -> None:
        # Cập nhật kéo dãn biên v (cực tiểu) và w (cực đại) để chứa điểm x
        x = np.asarray(x, dtype=float)
        self.v = np.minimum(self.v, x)
        self.w = np.maximum(self.w, x)
        self.expanded_count += 1

    def to_rule_string(
        self,
        feature_names: Optional[List[str]] = None,
        class_names: Optional[Dict] = None,
        x_min: Optional[np.ndarray] = None,
        x_max: Optional[np.ndarray] = None
    ) -> str:
        # Tạo chuỗi mô tả luật logic IF-THEN dễ đọc từ tọa độ hộp mờ
        if feature_names is None:
            feature_names = [f"Đặc trưng {i+1}" for i in range(self.n_features)]

        # Chuyển đổi ngược dữ liệu chuẩn hóa [0,1] về đơn vị thực tế (nếu có min/max)
        if x_min is not None and x_max is not None:
            real_v = self.v * (x_max - x_min) + x_min
            real_w = self.w * (x_max - x_min) + x_min
            fmt = ".2f"
        else:
            real_v = self.v
            real_w = self.w
            fmt = ".3f"

        conditions_list = []
        for i in range(self.n_features):
            cond = f"{real_v[i]:{fmt}} <= {feature_names[i]} <= {real_w[i]:{fmt}}"
            conditions_list.append(cond)

        label_text = class_names.get(self.class_label, str(self.class_label)) if class_names else str(self.class_label)
        return f"NẾU { ' VÀ '.join(conditions_list) } THÌ Kết quả = {label_text} (Hộp #{self.box_id}, N={self.expanded_count})"


# =============================================================================
# 2. BỘ PHÂN LOẠI MỜ FUZZY MIN-MAX (Học luật không giám sát một phần)
# =============================================================================
class FuzzyMinMaxClassifier:
    def __init__(
        self,
        theta: float = 0.2,
        gamma: float = 1.0,
        theta_per_class: Optional[Dict[Union[int, str], float]] = None,
        epsilon: float = 1e-12
    ):
        self.theta = theta
        self.gamma = gamma
        self.theta_per_class = theta_per_class
        self.epsilon = epsilon
        self.hyperboxes: List[Hyperbox] = []

    def _get_theta(self, label) -> float:
        # Tìm kích thước tối đa quy định riêng cho từng nhóm nhãn lớp học lực/cảnh báo
        if self.theta_per_class is not None and label in self.theta_per_class:
            return float(self.theta_per_class[label])
        return self.theta

    def _create_hyperbox(self, x: np.ndarray, label) -> Hyperbox:
        new_box = Hyperbox(box_id=len(self.hyperboxes), x=x, class_label=label)
        self.hyperboxes.append(new_box)
        return new_box

    def _find_overlap_dimension_and_case(self, box_j: Hyperbox, box_k: Hyperbox) -> Optional[Tuple[int, int]]:
        # Kiểm tra xem 2 hộp khác nhãn có bị chồng lấn chồng lấp đè lên nhau hay không
        n_features = box_j.n_features
        overlap_candidates = []

        for i in range(n_features):
            vj, wj = box_j.v[i], box_j.w[i]
            vk, wk = box_k.v[i], box_k.w[i]

            # Trường hợp không chồng lấp
            if wj < vk - self.epsilon or wk < vj - self.epsilon:
                return None
            if abs(wj - vk) <= self.epsilon and vj < vk - self.epsilon:
                return None
            if abs(wk - vj) <= self.epsilon and vk < vj - self.epsilon:
                return None

            # Xác định các trường hợp chồng chéo nhau
            if vj < vk - self.epsilon and vk <= wj + self.epsilon and wj < wk - self.epsilon:
                overlap_candidates.append((wj - vk, i, 1))
            elif vk < vj - self.epsilon and vj <= wk + self.epsilon and wk < wj - self.epsilon:
                overlap_candidates.append((wk - vj, i, 2))
            elif vj <= vk + self.epsilon and wk <= wj + self.epsilon and (wj - vj) >= (wk - vk) - self.epsilon:
                overlap_candidates.append((min(wk - vj, wj - vk), i, 3))
            elif vk <= vj + self.epsilon and wj <= wk + self.epsilon:
                overlap_candidates.append((min(wj - vk, wk - vj), i, 4))
            else:
                return None

        if len(overlap_candidates) == 0:
            return None

        # Sắp xếp để lấy kích thước chồng chéo nhỏ nhất cần co lại
        positive_candidates = [c for c in overlap_candidates if c[0] > self.epsilon]
        if len(positive_candidates) > 0:
            positive_candidates.sort(key=lambda t: t[0])
            _, delta, case = positive_candidates[0]
        else:
            overlap_candidates.sort(key=lambda t: t[0])
            _, delta, case = overlap_candidates[0]

        return delta, case

    def _contract(self, box_j: Hyperbox, box_k: Hyperbox, delta: int, case: int) -> None:
        # Co mép đường biên của 2 hộp lại tại chiều thứ delta để triệt tiêu vùng chồng chéo
        vj, wj = box_j.v[delta], box_j.w[delta]
        vk, wk = box_k.v[delta], box_k.w[delta]

        if case == 1:
            mid = (wj + vk) / 2.0
            box_j.w[delta] = mid
            box_k.v[delta] = mid
        elif case == 2:
            mid = (wk + vj) / 2.0
            box_k.w[delta] = mid
            box_j.v[delta] = mid
        elif case == 3:
            if abs(vj - vk) <= self.epsilon and abs(wj - wk) <= self.epsilon:
                mid = (vj + wj) / 2.0
                box_j.w[delta] = mid
                box_k.v[delta] = mid
            elif (wk - vj) < (wj - vk):
                box_j.v[delta] = wk
            else:
                box_j.w[delta] = vk
        elif case == 4:
            if abs(vj - vk) <= self.epsilon and abs(wj - wk) <= self.epsilon:
                mid = (vj + wj) / 2.0
                box_k.w[delta] = mid
                box_j.v[delta] = mid
            elif (wj - vk) < (wk - vj):
                box_k.v[delta] = wj
            else:
                box_k.w[delta] = vj

    def _resolve_overlap(self, box_j: Hyperbox, box_k: Hyperbox) -> bool:
        if box_j.class_label == box_k.class_label:
            return False
        overlap_result = self._find_overlap_dimension_and_case(box_j, box_k)
        if overlap_result is None:
            return False
        delta, case = overlap_result
        self._contract(box_j, box_k, delta, case)
        return True

    def fit(self, X: np.ndarray, y: np.ndarray) -> "FuzzyMinMaxClassifier":
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.hyperboxes = []

        # Huấn luyện mô hình bằng cách duyệt qua từng học sinh
        for x, label in zip(X, y):
            if len(self.hyperboxes) == 0:
                self._create_hyperbox(x, label)
                continue

            # Tìm các siêu hộp mờ có cùng nhãn lớp học lực với mẫu hiện tại
            same_class_boxes = []
            for box in self.hyperboxes:
                if box.class_label == label:
                    same_class_boxes.append(box)

            # Sắp xếp các hộp mờ cùng lớp theo thứ tự độ thuộc mờ từ cao xuống thấp
            if len(same_class_boxes) > 0:
                same_class_boxes.sort(key=lambda b: b.calculate_membership(x, self.gamma), reverse=True)

            expanded_box = None
            theta_label = self._get_theta(label)

            # Kiểm tra xem có mở rộng được hộp cũ nào không
            for box in same_class_boxes:
                if box.can_expand(x, theta_label, self.epsilon):
                    box.expand(x)
                    expanded_box = box
                    break

            # Nếu không tìm được hộp cũ phù hợp -> Tiến hành tạo hộp mờ mới cho sinh viên này
            if expanded_box is None:
                expanded_box = self._create_hyperbox(x, label)

            # Giải quyết chồng lấp giữa hộp mới mở rộng với các hộp của lớp khác
            for other_box in self.hyperboxes:
                if other_box.box_id != expanded_box.box_id:
                    self._resolve_overlap(expanded_box, other_box)
        return self

    def predict_one(self, x: np.ndarray):
        x = np.asarray(x, dtype=float)

        # 1. Tạo danh sách chứa độ khớp mờ của mẫu x đối với tất cả các hộp mờ
        membership_list = []
        for box in self.hyperboxes:
            score = box.calculate_membership(x, self.gamma)
            membership_list.append(score)

        # Chuyển thành mảng numpy để xử lý lọc nâng cao
        memberships = np.array(membership_list)
        max_mem = np.max(memberships)

        # Lọc ra danh sách các hộp mờ có độ thuộc đạt giá trị cao nhất (có tính sai số nhỏ)
        candidate_indices = []
        for idx in range(len(memberships)):
            if abs(memberships[idx] - max_mem) <= self.epsilon:
                candidate_indices.append(idx)

        # Nếu chỉ có duy nhất một hộp tối ưu nhất -> Gán nhãn học lực theo hộp đó
        if len(candidate_indices) == 1:
            best_box = self.hyperboxes[candidate_indices[0]]
        else:
            # Trường hợp hòa điểm mờ -> Tìm hộp có khoảng cách từ tâm đến x là ngắn nhất
            best_box_idx = candidate_indices[0]
            min_distance = float('inf')
            for idx in candidate_indices:
                dist = float(np.linalg.norm(self.hyperboxes[idx].center - x))
                if dist < min_distance:
                    min_distance = dist
                    best_box_idx = idx
            best_box = self.hyperboxes[best_box_idx]

        return best_box.class_label

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            return np.array([self.predict_one(X)])

        # Lặp tuần tự dự báo từng học sinh cho tập dữ liệu
        predictions = []
        for x in X:
            predictions.append(self.predict_one(x))
        return np.array(predictions)


# =============================================================================
# 3. PIPELINE LIÊN HOÀN (CASCADE MODEL) QUYẾT ĐỊNH CẢNH BÁO HỌC TẬP
# =============================================================================
class AcademicWarningPipeline:
    CLASS_MAPPING = {6: "Xuất sắc", 5: "Giỏi", 4: "Khá", 3: "Trung bình", 2: "Yếu", 1: "Kém"}
    WARNING_MAPPING = {0: "Bình thường", 1: "Cảnh báo học tập"}

    def __init__(self, theta_hl: float = 0.18, theta_cb: float = 0.12, random_state: int = 42):

        self.model_hocluc = FuzzyMinMaxClassifier(theta=theta_hl, gamma=1.0)
        # Giai đoạn cảnh báo: Sử dụng 2 phạm vi bao phủ khác nhau cho 2 nhãn 0 và 1
        self.model_canhbao = FuzzyMinMaxClassifier(
            theta=theta_cb,
            gamma=1.0,
            theta_per_class={0: theta_cb * 0.8, 1: theta_cb * 1.2}
        )
        self.x_min = np.array([0.0, 0.0, 0.0, 0.0, 1.0])
        self.x_max = np.array([4.0, 4.0, 141.0, 19.0, 7.0])
        self.feature_names = ['TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Kỳ xét']
        self.random_state = random_state

    def _fit_scaler(self, X_raw: np.ndarray):
        # Tìm ranh giới chuẩn hóa thực tế của tập dữ liệu huấn luyện
        data_min = np.min(X_raw, axis=0)
        data_max = np.max(X_raw, axis=0)
        self.x_min = np.array([0.0, 0.0, min(0.0, data_min[2]), min(0.0, data_min[3]), min(1.0, data_min[4])])
        self.x_max = np.array([4.0, 4.0, max(141.0, data_max[2]), max(19.0, data_max[3]), max(7.0, data_max[4])])

    def scale_features(self, X: np.ndarray) -> np.ndarray:
        # Chuẩn hóa giá trị đặc trưng về tầm phạm vi [0, 1] cho bài toán mờ
        X = np.asarray(X, dtype=float)
        scaled = (X - self.x_min) / (self.x_max - self.x_min + 1e-12)
        return np.clip(scaled, 0.0, 1.0)

    def fit(self, X_raw: np.ndarray, y_hocluc: np.ndarray, y_canhbao: np.ndarray):

        self._fit_scaler(X_raw)
        X_norm = self.scale_features(X_raw)

        # 1. Huấn luyện bước 1: Dự đoán xếp loại học lực của sinh viên
        self.model_hocluc.fit(X_norm, y_hocluc)

        # 2. Huấn luyện bước 2: Tập trung huấn luyện cảnh báo trên nhóm sinh viên học lực yếu kém
        idx_low_list = []
        for i in range(len(y_hocluc)):
            if y_hocluc[i] <= 2:
                idx_low_list.append(i)
        idx_low = np.array(idx_low_list)

        if len(idx_low) == 0:
            # Không có sinh viên yếu kém -> Không cần học nâng cao
            return self

        X_low = X_norm[idx_low]
        y_cb_low = y_canhbao[idx_low]

        idx_cb_list = []
        idx_norm_list = []
        for idx_pos, label in enumerate(y_cb_low):
            if label == 1:
                idx_cb_list.append(idx_pos)
            else:
                idx_norm_list.append(idx_pos)

        idx_cb = np.array(idx_cb_list)
        idx_norm = np.array(idx_norm_list)

        # Xử lý mất cân bằng lớp bằng phương pháp Jittering (Thêm nhiễu phân phối chuẩn nhỏ)
        if len(idx_cb) > 0:
            rng = np.random.default_rng(self.random_state)
            X_cb_orig = X_low[idx_cb]
            y_cb_orig = y_cb_low[idx_cb]

            X_jitter_list = [X_cb_orig]
            y_jitter_list = [y_cb_orig]
            noise_scale = np.array([0.015, 0.015, 0.010, 0.015, 0.0])

            for _ in range(5):
                noise = rng.normal(loc=0.0, scale=noise_scale, size=X_cb_orig.shape)
                X_aug = np.clip(X_cb_orig + noise, 0.0, 1.0)
                # Giữ nguyên thuộc tính 0 nếu ban đầu bằng 0
                for row_idx in range(len(X_cb_orig)):
                    if X_cb_orig[row_idx, 0] == 0.0:
                        X_aug[row_idx, 0] = 0.0
                    if X_cb_orig[row_idx, 3] == 0.0:
                        X_aug[row_idx, 3] = 0.0
                X_jitter_list.append(X_aug)
                y_jitter_list.append(y_cb_orig)

            # Ghép tập dữ liệu tổng hợp lại
            X_stage2 = np.vstack(X_jitter_list + [X_low[idx_norm]])
            y_stage2 = np.concatenate(y_jitter_list + [y_cb_low[idx_norm]])
        else:
            X_stage2, y_stage2 = X_low, y_cb_low

        #huấn luyện mô hình với tập dữ liệu bé
        self.model_canhbao.fit(X_stage2, y_stage2)
        return self

    def predict_student(self, tbc_hk, tbc_tl, tc_dat, tc_f, ky_xet, use_rule_fallback=True):
        sample_raw = np.array([[tbc_hk, tbc_tl, tc_dat, tc_f, ky_xet]], dtype=float)
        sample_norm = self.scale_features(sample_raw)

        # Dự đoán phân loại Học lực ở Giai đoạn 1
        pred_hl = int(self.model_hocluc.predict(sample_norm)[0])

        # Nếu xếp học lực <= 2 (Yếu/Kém) thì mới kích hoạt bộ mờ dự đoán cảnh báo học tập
        is_warning_fmnn = 0
        if pred_hl <= 2:
            is_warning_fmnn = int(self.model_canhbao.predict(sample_norm)[0])

        # Hệ thống luật dự phòng cứng (Fallback Rules)
        rule_warning = False
        if use_rule_fallback:
            is_deferred_leave = (tbc_hk == 0.0 and tc_f == 0 and tc_dat > 0)
            if not is_deferred_leave:
                cond_sem1 = (ky_xet == 1 and (tbc_hk < 0.80 or tc_f >= 9))
                cond_later_f = (ky_xet > 1 and tbc_hk < 1.00 and tc_f > 0)
                cond_later_active_low = (ky_xet > 1 and 0.0 < tbc_hk < 0.80)

                if cond_sem1 or cond_later_f or cond_later_active_low:
                    rule_warning = True

        final_warning = int(is_warning_fmnn or rule_warning)
        return {
            "pred_hocluc_code": pred_hl,
            "pred_hocluc_name": self.CLASS_MAPPING.get(pred_hl, "Không xác định"),
            "is_warning_fmnn": is_warning_fmnn,
            "is_warning": final_warning
        }

    def predict_batch(self, X_raw: np.ndarray, use_rule_fallback: bool = True):
        n_samples = len(X_raw)
        preds_hl = np.zeros(n_samples, dtype=int)
        preds_cb = np.zeros(n_samples, dtype=int)

        for i in range(n_samples):
            row = X_raw[i]
            res = self.predict_student(row[0], row[1], row[2], row[3], row[4], use_rule_fallback)
            preds_hl[i] = res["pred_hocluc_code"]
            preds_cb[i] = res["is_warning"]

        return preds_hl, preds_cb


In [6]:
import time
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

# 0. Tự động nạp dữ liệu đã làm sạch và chia sẵn 80% Train / 20% Test từ datasets/processed
try:
    _ = X_train, X_test, y_hl_train, y_cb_train, y_cb_test
    print("✅ Tìm thấy dữ liệu huấn luyện đã được nạp sẵn.")
except NameError:
    print("🔄 Đang nạp dữ liệu đã làm sạch và chia sẵn (80% Train / 20% Test) từ thư mục processed...")
    candidate_dirs = [
        processed_data_path if 'processed_data_path' in globals() else '',
        os.path.abspath(os.path.join(os.getcwd(), '..', 'datasets', 'processed')),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'FMN_EduWarning', 'datasets', 'processed')),
        os.path.abspath(os.path.join(os.getcwd(), 'FMN_EduWarning', 'datasets', 'processed')),
        os.path.abspath(os.path.join(os.getcwd(), 'datasets', 'processed')),
        '/content/drive/MyDrive/FMNN_CODE/FMN_EduWarning/datasets/processed'
    ]
    proc_dir = next((d for d in candidate_dirs if d and os.path.exists(d)), None)
    parquet_path = os.path.join(proc_dir, 'cleaned_data.parquet') if proc_dir else ''
    json_path = os.path.join(proc_dir, 'cleaned_data.json') if proc_dir else ''

    if parquet_path and os.path.exists(parquet_path):
        df = pd.read_parquet(parquet_path)
        print(f"📂 Đã nạp dữ liệu sạch từ Parquet: {parquet_path}")
    elif json_path and os.path.exists(json_path):
        df = pd.read_json(json_path, orient='records')
        print(f"📂 Đã nạp dữ liệu sạch từ JSON: {json_path}")
    else:
        raise FileNotFoundError("Không tìm thấy cleaned_data.parquet hoặc cleaned_data.json trong datasets/processed. Hãy chạy Tienxl.ipynb trước!")

    feature_cols = ['TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Kỳ xét']
    df_train = df[df['split'] == 'train']
    df_test = df[df['split'] == 'test']

    X_train = df_train[feature_cols].values
    X_test = df_test[feature_cols].values
    y_hl_train = df_train['Học lực'].values
    y_hl_test = df_test['Học lực'].values
    y_cb_train = df_train['Cảnh báo'].values
    y_cb_test = df_test['Cảnh báo'].values
    print(f"✅ Đã tách tự động theo cột 'split': Train = {len(X_train):,} mẫu (80%) | Test = {len(X_test):,} mẫu (20%)")

# 1. Đo thời gian thực thi (Huấn luyện & Dự báo)
try:
    start_time = time.time()
    pipeline_eval = AcademicWarningPipeline(theta_hl=0.18, theta_cb=0.12)
    pipeline_eval.fit(X_train, y_hl_train, y_cb_train)
    y_hl_pred_eval, y_cb_pred_eval = pipeline_eval.predict_batch(X_test)
    elapsed_time = time.time() - start_time

    # 2. Tính toán các chỉ số cho Cảnh báo (y_cb_test vs y_cb_pred_eval)
    acc_cb = accuracy_score(y_cb_test, y_cb_pred_eval)
    precision_cb, recall_cb, f1_cb, _ = precision_recall_fscore_support(y_cb_test, y_cb_pred_eval, average='macro', zero_division=0)

    # Tính toán Sensitivity và Specificity từ Confusion Matrix
    cm = confusion_matrix(y_cb_test, y_cb_pred_eval)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0,0], 0, 0, 0)
    sensitivity_cb = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity_cb = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # 3. Lấy số lượng Hyperboxes được tạo ra
    n_boxes_hl = len(pipeline_eval.model_hocluc.hyperboxes)
    n_boxes_cb = len(pipeline_eval.model_canhbao.hyperboxes)
    total_boxes = n_boxes_hl + n_boxes_cb

    # 4. Hiển thị bảng chỉ số chi tiết
    print("=" * 60)
    print("  BẢNG CHỈ SỐ ĐÁNH GIÁ CHI TIẾT MÔ HÌNH CẢNH BÁO (STAGE 2)")
    print("=" * 60)
    print(f"{'- Accuracy':<30}: {acc_cb * 100:.2f}%")
    print(f"{'- Precision (Macro)':<30}: {precision_cb * 100:.2f}%")
    print(f"{'- Recall (Macro)':<30}: {recall_cb * 100:.2f}%")
    print(f"{'- F1-Score (Macro)':<30}: {f1_cb * 100:.2f}%")
    print(f"{'- Sensitivity (Độ nhạy)':<30}: {sensitivity_cb * 100:.2f}%")
    print(f"{'- Specificity (Độ đặc hiệu)':<30}: {specificity_cb * 100:.2f}%")
    print(f"{'- Hyperbox Count (Học lực)':<30}: {n_boxes_hl} boxes")
    print(f"{'- Hyperbox Count (Cảnh báo)':<30}: {n_boxes_cb} boxes")
    print(f"{'- Total Hyperboxes':<30}: {total_boxes} boxes")
    print(f"{'- Execution Time':<30}: {elapsed_time:.4f} seconds")
    print("=" * 60)
except NameError as e:
    print(f"❌ Lỗi huấn luyện: {e}. Vui lòng đảm bảo bạn đã định nghĩa các hàm hoặc lớp AcademicWarningPipeline ở các bước trước.")

✅ Tìm thấy dữ liệu huấn luyện đã được nạp sẵn.
  BẢNG CHỈ SỐ ĐÁNH GIÁ CHI TIẾT MÔ HÌNH CẢNH BÁO (STAGE 2)
- Accuracy                    : 99.73%
- Precision (Macro)           : 93.67%
- Recall (Macro)              : 99.22%
- F1-Score (Macro)            : 96.27%
- Sensitivity (Độ nhạy)       : 98.70%
- Specificity (Độ đặc hiệu)   : 99.74%
- Hyperbox Count (Học lực)    : 166 boxes
- Hyperbox Count (Cảnh báo)   : 99 boxes
- Total Hyperboxes            : 265 boxes
- Execution Time              : 27.2740 seconds


In [5]:
import time
from sklearn.metrics import precision_recall_fscore_support

# 1. Đo thời gian thực thi (Huấn luyện & Dự báo)
start_time = time.time()
pipeline_eval = AcademicWarningPipeline(theta_hl=0.18, theta_cb=0.12)
pipeline_eval.fit(X_train, y_hl_train, y_cb_train)
y_hl_pred_eval, y_cb_pred_eval = pipeline_eval.predict_batch(X_test)
elapsed_time = time.time() - start_time

# 2. Tính toán các chỉ số cho Cảnh báo (y_cb_test vs y_cb_pred_eval)
acc_cb = accuracy_score(y_cb_test, y_cb_pred_eval)
pand_metrics = precision_recall_fscore_support(y_cb_test, y_cb_pred_eval, average='macro', zero_division=0)
precision_cb, recall_cb, f1_cb, _ = pand_metrics

# Tính toán Sensitivity và Specificity từ Confusion Matrix cho bài toán phân loại nhị phân (Cảnh báo)
cm = confusion_matrix(y_cb_test, y_cb_pred_eval)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0,0], 0, 0, 0) # Phòng trường hợp chỉ có 1 lớp trong tập test
sensitivity_cb = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_cb = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# 3. Lấy số lượng Hyperboxes được tạo ra
n_boxes_hl = len(pipeline_eval.model_hocluc.hyperboxes)
n_boxes_cb = len(pipeline_eval.model_canhbao.hyperboxes)
total_boxes = n_boxes_hl + n_boxes_cb

# 4. Hiển thị bảng chỉ số chi tiết
print("=" * 60)
print("  BẢNG CHỈ SỐ ĐÁNH GIÁ CHI TIẾT MÔ HÌNH CẢNH BÁO (STAGE 2)")
print("=" * 60)
print(f"{'- Accuracy':<30}: {acc_cb * 100:.2f}%")
print(f"{'- Precision (Macro)':<30}: {precision_cb * 100:.2f}%")
print(f"{'- Recall (Macro)':<30}: {recall_cb * 100:.2f}%")
print(f"{'- F1-Score (Macro)':<30}: {f1_cb * 100:.2f}%")
print(f"{'- Sensitivity (Độ nhạy)':<30}: {sensitivity_cb * 100:.2f}%")
print(f"{'- Specificity (Độ đặc hiệu)':<30}: {specificity_cb * 100:.2f}%")
print(f"{'- Hyperbox Count (Học lực)':<30}: {n_boxes_hl} boxes")
print(f"{'- Hyperbox Count (Cảnh báo)':<30}: {n_boxes_cb} boxes")
print(f"{'- Total Hyperboxes':<30}: {total_boxes} boxes")
print(f"{'- Execution Time':<30}: {elapsed_time:.4f} seconds")
print("=" * 60)

  BẢNG CHỈ SỐ ĐÁNH GIÁ CHI TIẾT MÔ HÌNH CẢNH BÁO (STAGE 2)
- Accuracy                    : 99.73%
- Precision (Macro)           : 93.67%
- Recall (Macro)              : 99.22%
- F1-Score (Macro)            : 96.27%
- Sensitivity (Độ nhạy)       : 98.70%
- Specificity (Độ đặc hiệu)   : 99.74%
- Hyperbox Count (Học lực)    : 166 boxes
- Hyperbox Count (Cảnh báo)   : 99 boxes
- Total Hyperboxes            : 265 boxes
- Execution Time              : 23.2075 seconds


In [ ]:
import numpy as np

def predict_single_student_interactive():
    model_pipeline = pipeline_eval if 'pipeline_eval' in globals() else pipeline

    print("=" * 85)
    print("  DANH SÁCH CHI TIẾT SIÊU HỘP MỜ CẢNH BÁO (STAGE 2 - CANH BAO) THEO TỪNG CHIỀU")
    print("=" * 85)

    warning_boxes = [box for box in model_pipeline.model_canhbao.hyperboxes if box.class_label == 1]

    if not warning_boxes:
        print("Không tìm thấy hộp siêu không gian mờ nào quy định nhãn Cảnh báo.")
    else:
        for idx, box in enumerate(warning_boxes, 1):
            # Giải chuẩn hóa tọa độ hộp để hiển thị thực tế
            v_real = box.v * (model_pipeline.x_max - model_pipeline.x_min) + model_pipeline.x_min
            w_real = box.w * (model_pipeline.x_max - model_pipeline.x_min) + model_pipeline.x_min

            print(f"📦 Hộp mờ #{idx} (ID Hộp: {box.box_id}, Số mẫu phủ N={box.expanded_count}):")
            for i, f_name in enumerate(model_pipeline.feature_names):
                print(f"   • Chiều '{f_name:<10}': Khoảng mờ [tối thiểu: {v_real[i]:.2f} -> tối đa: {w_real[i]:.2f}]")
            print("-" * 85)

    print("\n" + "=" * 85)
    print("    NHẬP THÔNG TIN SINH VIÊN MỚI ĐỂ PHÂN TÍCH QUYẾT ĐỊNH")
    print("=" * 85)

    try:
        tbc_hk = float(input("1. Điểm TBC học kỳ (hệ 4.0): "))
        tbc_tl = float(input("2. Điểm TBC tích lũy (hệ 4.0): "))
        tc_dat = int(input("3. Số tín chỉ đạt tích lũy: "))
        tc_f = int(input("4. Số tín chỉ bị điểm F trong kỳ: "))
        ky_xet = int(input("5. Kỳ xét hiện tại (ví dụ: 1, 2, ..., 7): "))

        # Chuẩn hóa mẫu đầu vào để tính toán độ thuộc mờ
        sample_raw = np.array([[tbc_hk, tbc_tl, tc_dat, tc_f, ky_xet]], dtype=float)
        sample_norm = model_pipeline.scale_features(sample_raw)[0]

        # Thực hiện dự báo qua Pipeline
        result = model_pipeline.predict_student(tbc_hk, tbc_tl, tc_dat, tc_f, ky_xet, use_rule_fallback=True)

        print("\n" + "=" * 85)
        print("          PHÂN TÍCH QUYẾT ĐỊNH & ĐỘ KHỚP SIÊU HỘP MỜ")
        print("=" * 85)
        print(f"[*] Dự báo xếp loại Học lực (Stage 1): {result['pred_hocluc_name']} (Mã nhãn: {result['pred_hocluc_code']})")

        # Phân tích độ thuộc mờ (membership) của sinh viên đối với các hộp cảnh báo
        if result['pred_hocluc_code'] <= 2:
            print("[*] Sinh viên thuộc nhóm học lực kém (<= Yếu) -> Được chuyển sang Stage 2 (Xét cảnh báo)")
            memberships = [box.calculate_membership(sample_norm, model_pipeline.model_canhbao.gamma) for box in warning_boxes]

            if warning_boxes:
                best_idx = int(np.argmax(memberships))
                best_box = warning_boxes[best_idx]
                max_mem = memberships[best_idx]

                print("\n--- Đo mức độ khớp (Membership) đối với các siêu hộp cảnh báo ---")
                for idx, (box, mem) in enumerate(zip(warning_boxes, memberships), 1):
                    star = "⭐ (HỘP QUYẾT ĐỊNH ĐƯỢC CHỌN)" if box.box_id == best_box.box_id else ""
                    print(f"   • Độ khớp với Hộp #{idx}: {mem * 100:.2f}% {star}")

                print(f"\n👉 Siêu hộp mờ có tầm ảnh hưởng lớn nhất: Hộp #{best_idx+1}")
                rule_str = best_box.to_rule_string(
                    feature_names=model_pipeline.feature_names,
                    class_names={0: "Bình thường", 1: "Cảnh báo học tập"},
                    x_min=model_pipeline.x_min,
                    x_max=model_pipeline.x_max
                )
                print(f"   Luật mờ kích hoạt: {rule_str}")
        else:
            print("[*] Sinh viên thuộc nhóm học lực an toàn (> Yếu) -> Bỏ qua giai đoạn xét siêu hộp cảnh báo ở Stage 2.")

        # Phân tích hệ thống luật dự phòng bổ sung (Fallback Rules)
        print("\n--- Kiểm tra luật dự phòng (Rule-based Fallback) ---")
        cond_sem1 = (ky_xet == 1 and (tbc_hk < 0.80 or tc_f >= 9))
        cond_later_f = (ky_xet > 1 and tbc_hk < 1.00 and tc_f > 0)
        cond_later_active_low = (ky_xet > 1 and 0.0 < tbc_hk < 0.80)
        is_deferred_leave = (tbc_hk == 0.0 and tc_f == 0 and tc_dat > 0)

        print(f"   • Luật Kỳ 1 (TBC HK < 0.8 hoặc TC F >= 9): {'Bị kích hoạt' if cond_sem1 else 'Không kích hoạt'}")
        print(f"   • Luật Kỳ sau (TBC HK < 1.0 và có TC F): {'Bị kích hoạt' if cond_later_f else 'Không kích hoạt'}")
        print(f"   • Luật Học lực quá kém ở kỳ sau (0.0 < TBC < 0.8): {'Bị kích hoạt' if cond_later_active_low else 'Không kích hoạt'}")
        print(f"   • Trạng thái tạm hoãn/Nghỉ học tạm thời: {'Hợp lệ' if is_deferred_leave else 'Không có'}")

        # Kết luận cuối cùng
        print("\n" + "=" * 85)
        print("                                KẾT LUẬN CUỐI CÙNG")
        print("=" * 85)
        print(f"• Cảnh báo học tập từ bộ FMNN    : {'⚠️ CẢNH BÁO' if result['is_warning_fmnn'] == 1 else '✅ BÌNH THƯỜNG'}")
        print(f"• Kết luận cuối cùng (Kết hợp luật): {'⚠️ CẢNH BÁO HỌC TẬP' if result['is_warning'] == 1 else '✅ BÌNH THƯỜNG'}")
        print("=" * 85)

    except ValueError:
        print("❌ Lỗi: Vui lòng nhập đúng định dạng số cho các trường dữ liệu!")
    except NameError:
        print("❌ Lỗi: Mô hình pipeline chưa được khởi tạo. Hãy chạy các cell huấn luyện ở phía trên trước!")

# Khởi chạy giao diện giám sát
predict_single_student_interactive()

  DANH SÁCH CHI TIẾT SIÊU HỘP MỜ CẢNH BÁO (STAGE 2 - CANH BAO) THEO TỪNG CHIỀU
📦 Hộp mờ #1 (ID Hộp: 0, Số mẫu phủ N=12):
   • Chiều 'TBC HK    ': Khoảng mờ [tối thiểu: 0.00 -> tối đa: 0.00]
   • Chiều 'TBC TL    ': Khoảng mờ [tối thiểu: 0.00 -> tối đa: 0.05]
   • Chiều 'Số TC Đạt ': Khoảng mờ [tối thiểu: 0.00 -> tối đa: 1.72]
   • Chiều 'Số TC F   ': Khoảng mờ [tối thiểu: 0.00 -> tối đa: 0.00]
   • Chiều 'Kỳ xét    ': Khoảng mờ [tối thiểu: 2.00 -> tối đa: 2.00]
-------------------------------------------------------------------------------------

    NHẬP THÔNG TIN SINH VIÊN MỚI ĐỂ PHÂN TÍCH QUYẾT ĐỊNH
1. Điểm TBC học kỳ (hệ 4.0): 2.05
2. Điểm TBC tích lũy (hệ 4.0): 2.55
3. Số tín chỉ đạt tích lũy: 32
4. Số tín chỉ bị điểm F trong kỳ: 3
5. Kỳ xét hiện tại (ví dụ: 1, 2, ..., 7): 2

          PHÂN TÍCH QUYẾT ĐỊNH & ĐỘ KHỚP SIÊU HỘP MỜ
[*] Dự báo xếp loại Học lực (Stage 1): Không xác định (Mã nhãn: 0)
[*] Sinh viên thuộc nhóm học lực kém (<= Yếu) -> Được chuyển sang Stage 2 (Xét cảnh báo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Lưu ý khi chạy trên Google Colab:

1. Đảm bảo bạn đã upload file `ket_qua_xet_len_lop_ktpm.csv` vào đúng thư mục Google Drive: `/content/drive/MyDrive/FMNN_CODE/datasets/raw/`.


2. Bạn có thể copy toàn bộ đoạn mã trên vào một cell mới trên Google Colab và bấm **Run** để tiến hành huấn luyện và đánh giá toàn bộ mô hình.